# 02 — CLIP Fine-tuning on BigEarthNet

This notebook walks through fine-tuning OpenCLIP on BigEarthNet for remote sensing scene understanding.

## Steps
1. Load pretrained OpenCLIP (ViT-B/32)
2. Prepare BigEarthNet data loaders
3. Run contrastive fine-tuning
4. Evaluate zero-shot retrieval
5. Visualise embedding space


In [ ]:
import sys
sys.path.insert(0, '../backend')

import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import open_clip
import torch.nn.functional as F
from torch.utils.data import DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = os.environ.get('BIGEARTHNET_DIR', '/data/BigEarthNet')

print(f'Device: {DEVICE}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Load OpenCLIP ─────────────────────────────────────────────────────────────
MODEL_NAME = 'ViT-B-32'
PRETRAINED = 'openai'

model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model = model.to(DEVICE)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Model: {MODEL_NAME}  |  Parameters: {n_params:.1f}M')

In [ ]:
# ── Data loaders ─────────────────────────────────────────────────────────────
from training.bigearthnet_dataset import BigEarthNetDataset

BATCH_SIZE = 32
MAX_SAMPLES = 1000  # Reduce for quick demo

train_ds = BigEarthNetDataset(DATA_DIR, split='train', use_sar=False, max_samples=MAX_SAMPLES)
val_ds   = BigEarthNetDataset(DATA_DIR, split='val',   use_sar=False, max_samples=MAX_SAMPLES // 5)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

In [ ]:
# ── Quick zero-shot evaluation BEFORE fine-tuning ─────────────────────────────
from training.evaluate import CLIPRetrievalEvaluator, BIGEARTHNET_43_LABELS

evaluator = CLIPRetrievalEvaluator(model, tokenizer, DEVICE)
baseline = evaluator.evaluate(val_loader, BIGEARTHNET_43_LABELS)

print('=== Baseline (pretrained, no fine-tuning) ===')
for group, scores in baseline.items():
    print(f'  {group}:')
    for k, v in scores.items():
        print(f'    {k}: {v:.2f}%')

In [ ]:
# ── Fine-tuning loop (mini version) ──────────────────────────────────────────
from training.finetune_clip import train_one_epoch, evaluate, contrastive_loss
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS = 3  # Increase for real training
LR = 1e-5

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.1)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

train_losses = []
val_r1_scores = []

for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch(model, tokenizer, train_loader, optimizer, DEVICE, epoch, log_interval=10)
    scheduler.step()
    metrics = evaluate(model, tokenizer, val_loader, DEVICE)

    train_losses.append(loss)
    val_r1_scores.append(metrics['R@1'])

    print(f'Epoch {epoch}/{EPOCHS} | Loss: {loss:.4f} | R@1: {metrics["R@1"]:.2f}% | R@5: {metrics["R@5"]:.2f}%')

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#0a1020')

for ax in (ax1, ax2):
    ax.set_facecolor('#0f1a30')
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_color('#334155')

ax1.plot(range(1, EPOCHS+1), train_losses, 'o-', color='#3aabff', linewidth=2)
ax1.set_title('Training Loss', color='white')
ax1.set_xlabel('Epoch', color='white')
ax1.set_ylabel('Contrastive Loss', color='white')

ax2.plot(range(1, EPOCHS+1), val_r1_scores, 's-', color='#22c55e', linewidth=2)
ax2.set_title('Val Retrieval R@1 (%)', color='white')
ax2.set_xlabel('Epoch', color='white')
ax2.set_ylabel('R@1 (%)', color='white')

plt.tight_layout()
plt.show()

In [ ]:
# ── Embedding visualisation (t-SNE) ──────────────────────────────────────────
from sklearn.manifold import TSNE

model.eval()
all_feats = []
all_labels_names = []
from training.bigearthnet_dataset import BIGEARTHNET_43_LABELS

with torch.no_grad():
    for optical, _, labels_batch, _ in val_loader:
        rgb = optical[:, [3, 2, 1], :, :].to(DEVICE)
        rgb = torch.clamp((rgb * 0.2 + 0.5), 0.0, 1.0)
        feats = F.normalize(model.encode_image(rgb), dim=-1).cpu().numpy()
        all_feats.append(feats)

        for row in labels_batch:
            lbls = [BIGEARTHNET_43_LABELS[i] for i, v in enumerate(row) if v > 0.5]
            all_labels_names.append(lbls[0] if lbls else 'Unknown')

        if len(all_feats) * BATCH_SIZE > 500:
            break

feats = np.concatenate(all_feats)
print(f'Computing t-SNE on {feats.shape[0]} embeddings...')

tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=500)
emb_2d = tsne.fit_transform(feats)

# Plot
unique_labels = list(set(all_labels_names))[:15]
colors = plt.cm.tab20(np.linspace(0, 1, len(unique_labels)))
color_map = {lbl: colors[i] for i, lbl in enumerate(unique_labels)}

fig, ax = plt.subplots(figsize=(12, 8))
fig.patch.set_facecolor('#0a1020')
ax.set_facecolor('#0f1a30')

for lbl in unique_labels:
    mask = [n == lbl for n in all_labels_names[:len(emb_2d)]]
    pts = emb_2d[mask]
    if len(pts):
        ax.scatter(pts[:, 0], pts[:, 1], c=[color_map[lbl]], label=lbl, s=15, alpha=0.7)

ax.legend(loc='upper right', fontsize=7, facecolor='#0f1a30', labelcolor='white',
          edgecolor='#334155', framealpha=0.8)
ax.set_title('t-SNE of Fine-tuned CLIP Embeddings (BigEarthNet)', color='white', fontsize=12)
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_color('#334155')

plt.tight_layout()
plt.show()